# Lesson 02 - Khám phá Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** là một khung thống nhất để xây dựng các tác nhân AI. Nó cung cấp một kiến trúc rõ ràng, có thể kết hợp với bốn khối xây dựng cốt lõi:

- **Client** – kết nối với điểm cuối mô hình AI và xử lý giao tiếp
- **Agent** – bao bọc một client với các hướng dẫn và định nghĩa công cụ
- **Tools** – mở rộng khả năng của tác nhân với các chức năng tùy chỉnh mà mô hình có thể gọi
- **Session** – duy trì lịch sử trò chuyện cho các tương tác đa lượt

Trong bài học này, chúng ta sẽ xây dựng một **tác nhân đặt chuyến đi** kiểm tra khả năng có sẵn của điểm đến sử dụng các khái niệm này.


## Cài đặt


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Hiểu về Kiến trúc Khung Agent

Khung Agent của Microsoft tuân theo kiến trúc phân lớp:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Một `AzureAIProjectAgentProvider` kết nối với một triển khai Azure OpenAI. Nó xử lý việc xác thực, định dạng yêu cầu và phân tích phản hồi.
2. **Agent** – Được tạo ra từ client thông qua `provider.create_agent()`, agent kết hợp truy cập mô hình với các hướng dẫn (lời nhắc hệ thống) và công cụ.
3. **Tools** – Các hàm Python được trang trí với `@tool` mà agent có thể gọi để thực hiện hành động hoặc truy xuất dữ liệu.
4. **Session** – Một đối tượng `AgentSession` (được tạo qua `agent.create_session()`) lưu trữ lịch sử hội thoại, cho phép đối thoại đa lượt mà agent nhớ bối cảnh trước đó.

Hãy cùng xây dựng từng lớp một từng bước.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Thêm Công cụ với Bộ trang trí @tool

Công cụ cho phép các tác nhân thực hiện các hành động vượt ra ngoài việc tạo văn bản. Bộ trang trí `@tool` chuyển một hàm Python thông thường thành một thứ mà tác nhân có thể gọi.

Các điểm chính:
- Sử dụng `Annotated[type, "description"]` để mô hình hiểu từng tham số.
- Chuỗi tài liệu trở thành mô tả công cụ mà mô hình nhìn thấy.
- `approval_mode="never_require"` nghĩa là công cụ sẽ chạy tự động mà không cần xác nhận của người dùng.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Tạo một Agent với Công cụ

Bây giờ chúng ta kết hợp client, hướng dẫn, và công cụ thành một agent. `instructions` hoạt động như lời nhắc hệ thống — chúng xác định tính cách và hành vi của agent.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Đàm Thoại Nhiều Lượt với Phiên

Một `AgentSession` (được tạo qua `agent.create_session()`) theo dõi tất cả các tin nhắn trong một cuộc trò chuyện. Bằng cách truyền cùng một phiên đến mỗi lần gọi `agent.run()`, tác nhân có thể truy cập toàn bộ lịch sử cuộc trò chuyện và có thể tham chiếu lại các tin nhắn trước đó.

Chúng tôi truyền `tools=[check_destination_availability]` để tác nhân có thể gọi công cụ kiểm tra khả dụng của chúng tôi trong mỗi lượt.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Tóm tắt

Trong bài học này, bạn đã khám phá bốn trụ cột của Microsoft Agent Framework:

| Khái niệm | Những gì bạn đã học |
|---------|------------------|
| **Khách hàng** | `AzureAIProjectAgentProvider` kết nối với Azure OpenAI bằng xác thực dựa trên thông tin đăng nhập |
| **Agent** | `provider.create_agent()` gói kết nối mô hình với hướng dẫn và tên |
| **Công cụ** | Trình trang trí `@tool` cho phép các hàm Python được agent gọi |
| **Phiên** | `agent.create_session()` duy trì lịch sử cuộc trò chuyện qua nhiều lượt |

Những thành phần cơ bản này kết hợp với nhau để tạo ra các agent có thể giữ cuộc trò chuyện tự nhiên, gọi các hàm bên ngoài, và duy trì ngữ cảnh — nền tảng cho các mẫu agent nâng cao hơn trong các bài học sau.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:  
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sự không chính xác. Văn bản gốc bằng ngôn ngữ gốc nên được xem là nguồn chính thức và đáng tin cậy. Đối với các thông tin quan trọng, việc sử dụng bản dịch chuyên nghiệp do con người thực hiện được khuyến nghị. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu lầm hay giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
